# 04 — Batch Scoring

**Purpose:** Load the champion model from MLflow Model Registry,
score all materials in the Gold feature table,
compute SHAP-based top-3 driver explanations per material,
and write the results to two serving tables.

**Tables produced:**
| Table | Description |
|---|---|
| `ss_recommendations` | Model output: current SS, new SS, % change, SHAP drivers, confidence |
| `approval_requests` | Empty schema ready to receive buyer submissions |

**Prerequisite:** Run `03_train_model` first.

## 0. Configuration — Edit before running

In [ ]:
CATALOG    = 'dev'
SCHEMA     = 'safety_stock_gold'
MODEL_NAME = 'safety-stock-model'
EXPERIMENT = 'safety-stock-estimation'

FEATURE_COLS = [
    'demand_mean', 'demand_std', 'demand_cv',
    'lead_time_mean', 'lead_time_std',
    'service_level_z',
    'abc_class_encoded', 'category_encoded',
]

DRIVER_LABELS = {
    'demand_mean':        'avg demand',
    'demand_std':         'demand variability (\u03c3)',
    'demand_cv':          'demand CV',
    'lead_time_mean':     'avg lead time',
    'lead_time_std':      'lead time variability',
    'service_level_z':    'service level target',
    'abc_class_encoded':  'ABC classification',
    'category_encoded':   'material category',
}

spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'\u2705 Using {CATALOG}.{SCHEMA}')
print(f'Model: {MODEL_NAME}')

✅ Using dev.safety_stock_gold
Model: safety-stock-model


## 1. Install ML Libraries (if not already on cluster)

In [ ]:
%pip install scikit-learn shap --quiet

## 2. Imports

In [ ]:
import uuid
import numpy as np
import pandas as pd
import shap
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

print('\u2705 Imports ready')

✅ Imports ready


## 3. Load Gold Features

In [ ]:
gold_pdf = spark.table(f'{CATALOG}.{SCHEMA}.safety_stock_features').toPandas()
print(f'Gold features loaded: {len(gold_pdf)} materials')

Gold features loaded: 100 materials


## 4. Load Champion Model from MLflow Registry

In [ ]:
mlflow.set_experiment(EXPERIMENT)
client   = mlflow.tracking.MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest   = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
model_uri = f'models:/{MODEL_NAME}/{latest.version}'

print(f'Loading: {model_uri}')
model = mlflow.sklearn.load_model(model_uri)
print('\u2705 Model loaded')

Loading: models:/safety-stock-model/1
✅ Model loaded


## 5. Score Materials + Confidence Interval

In [ ]:
X = gold_pdf[FEATURE_COLS].fillna(0)

print(f'Scoring {len(gold_pdf)} materials...')
new_ss = np.round(model.predict(X)).clip(1).astype(int)

# Confidence: use std dev across trees (lower std = higher confidence)
tree_preds = np.array([t.predict(X) for t in model.estimators_])
pred_std   = tree_preds.std(axis=0)
max_std    = pred_std.max() if pred_std.max() > 0 else 1.0
confidence = (1 - pred_std / max_std).clip(0, 1).round(3)

print('\u2705 Scoring complete')

Scoring 100 materials...
✅ Scoring complete


## 6. SHAP Explanations — Top 3 Drivers per Material

In [ ]:
print('Computing SHAP values...')
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

driver_rows = []
for i, row_shap in enumerate(shap_values):
    abs_shap = np.abs(row_shap)
    top_idx  = np.argsort(abs_shap)[::-1][:3]
    drivers  = []
    for idx in top_idx:
        feat      = FEATURE_COLS[idx]
        direction = '\u2191' if row_shap[idx] > 0 else '\u2193'
        label     = DRIVER_LABELS.get(feat, feat)
        val       = X.iloc[i][feat]
        drivers.append(f'{direction} {label} ({val:.2f})')
    driver_rows.append({
        'driver_1': drivers[0] if len(drivers) > 0 else '',
        'driver_2': drivers[1] if len(drivers) > 1 else '',
        'driver_3': drivers[2] if len(drivers) > 2 else '',
    })

drivers_pdf = pd.DataFrame(driver_rows)
print('\u2705 SHAP explanations ready')

Computing SHAP values...
✅ SHAP explanations ready


## 7. Assemble Recommendations Table

In [ ]:
recs_pdf = gold_pdf[[
    'material_id', 'plant', 'buyer_id', 'material_desc', 'category', 'abc_class', 'current_ss'
]].copy().reset_index(drop=True)

recs_pdf['new_ss']           = new_ss
recs_pdf['pct_change']       = ((recs_pdf['new_ss'] - recs_pdf['current_ss']) /
                                 recs_pdf['current_ss'].clip(lower=1) * 100).round(1)
recs_pdf['confidence_score'] = confidence
recs_pdf = pd.concat([recs_pdf, drivers_pdf], axis=1)
recs_pdf['scored_at']        = pd.Timestamp.now()
recs_pdf['status']           = 'pending_review'

n_up     = (recs_pdf['pct_change'] >  5).sum()
n_down   = (recs_pdf['pct_change'] < -5).sum()
n_stable = len(recs_pdf) - n_up - n_down
print(f'SS increase (> 5%) : {n_up:>3} materials')
print(f'SS decrease (< -5%): {n_down:>3} materials')
print(f'SS stable          : {n_stable:>3} materials')

SS increase (> 5%) :  38 materials
SS decrease (< -5%):  29 materials
SS stable          :  33 materials


## 8. Write Serving Tables

In [ ]:
def write_table(sdf, table_name: str, comment: str) -> None:
    (
        sdf.write
           .format('delta')
           .mode('overwrite')
           .option('overwriteSchema', 'true')
           .saveAsTable(f'{CATALOG}.{SCHEMA}.{table_name}')
    )
    safe = comment.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{SCHEMA}.{table_name} IS '{safe}'")
    count = spark.table(f'{CATALOG}.{SCHEMA}.{table_name}').count()
    print(f'  \u2705  {CATALOG}.{SCHEMA}.{table_name}  \u2192  {count:,} rows')

print('Writing serving tables...')

# ── ss_recommendations ───────────────────────────────────────────────────────
recs_sdf = spark.createDataFrame(recs_pdf)
write_table(
    recs_sdf, 'ss_recommendations',
    'ML-generated safety stock recommendations. One row per material. '
    'Contains current SS, new SS, % change, confidence score, and top-3 SHAP drivers. '
    'Populated by batch scoring notebook and consumed by the Buyer Dashboard.'
)

# ── approval_requests (empty schema) ─────────────────────────────────────────
approval_schema = StructType([
    StructField('request_id',      StringType(),    True),
    StructField('material_id',     StringType(),    True),
    StructField('plant',           StringType(),    True),
    StructField('buyer_id',        StringType(),    True),
    StructField('current_ss',      IntegerType(),   True),
    StructField('new_ss',          IntegerType(),   True),
    StructField('pct_change',      DoubleType(),    True),
    StructField('driver_1',        StringType(),    True),
    StructField('driver_2',        StringType(),    True),
    StructField('driver_3',        StringType(),    True),
    StructField('status',          StringType(),    True),
    StructField('submitted_at',    TimestampType(), True),
    StructField('manager_id',      StringType(),    True),
    StructField('reviewed_at',     TimestampType(), True),
    StructField('manager_comment', StringType(),    True),
])
empty_sdf = spark.createDataFrame([], approval_schema)
write_table(
    empty_sdf, 'approval_requests',
    'Approval workflow state table. One row per buyer-submitted SS change request. '
    'Status values: pending | approved | rejected. '
    'Populated by the Buyer Dashboard and actioned by the Manager Approval page.'
)

print('\n\U0001F389 Batch scoring complete.')

Writing serving tables...
  ✅  dev.safety_stock_gold.ss_recommendations  →  100 rows
  ✅  dev.safety_stock_gold.approval_requests   →  0 rows (empty schema)

🎉 Batch scoring complete.


## 9. Verify

In [ ]:
tables = ['ss_recommendations', 'approval_requests']

print(f"{'Table':<35} {'Row count':>10}")
print('-' * 47)
for t in tables:
    count = spark.table(f'{CATALOG}.{SCHEMA}.{t}').count()
    print(f'{t:<35} {count:>10,}')

Table                              Row count
-------------------------------------------------
ss_recommendations                       100
approval_requests                          0


### Sample recommendations

In [ ]:
display(
    spark.table(f'{CATALOG}.{SCHEMA}.ss_recommendations')
        .select('material_id', 'abc_class', 'current_ss', 'new_ss',
                'pct_change', 'confidence_score', 'driver_1')
        .orderBy(F.abs(F.col('pct_change')).desc())
        .limit(10)
)

## 10. Add Column-Level Comments

In [ ]:
col_comments = {
    'ss_recommendations': {
        'material_id':      'Material identifier. Primary key for this table.',
        'plant':            'Plant code.',
        'buyer_id':         'Buyer responsible for this material.',
        'current_ss':       'Current safety stock in SAP before the ML recommendation.',
        'new_ss':           'ML-recommended safety stock (RandomForest prediction, rounded to integer).',
        'pct_change':       'Percentage change from current_ss to new_ss. Positive = increase.',
        'confidence_score': 'Model confidence 0–1 (1 = low variance across trees, i.e. high confidence).',
        'driver_1':         'Top SHAP driver with direction (\u2191 = pushes SS up, \u2193 = pushes SS down).',
        'driver_2':         'Second SHAP driver.',
        'driver_3':         'Third SHAP driver.',
        'scored_at':        'Timestamp when the batch scoring was run.',
        'status':           'Review status: pending_review | submitted | approved | rejected.',
    },
    'approval_requests': {
        'request_id':      'UUID for this approval request. Primary key.',
        'material_id':     'Material being changed.',
        'plant':           'Plant code.',
        'buyer_id':        'Buyer who submitted this request.',
        'current_ss':      'Safety stock before the change.',
        'new_ss':          'Proposed new safety stock.',
        'pct_change':      'Percentage change.',
        'status':          'Workflow state: pending | approved | rejected.',
        'submitted_at':    'Timestamp when the buyer submitted the request.',
        'manager_id':      'Manager who reviewed the request.',
        'reviewed_at':     'Timestamp when the manager approved or rejected.',
        'manager_comment': 'Optional comment left by the manager.',
    },
}

for table, cols in col_comments.items():
    for col, comment in cols.items():
        safe = comment.replace("'", "''")
        spark.sql(f'ALTER TABLE {CATALOG}.{SCHEMA}.{table} ALTER COLUMN {col} COMMENT \'{safe}\'')
    print(f'  \u2705  Column comments added \u2192 {table}')

print('\n\u2705 All column-level comments applied.')

  ✅  Column comments added → ss_recommendations
  ✅  Column comments added → approval_requests

✅ All column-level comments applied.


## Next Step

The pipeline is complete. Launch the Streamlit app:

```bash
streamlit run app/streamlit_app.py
```

Make sure `.env` contains your `DATABRICKS_HOST`, `DATABRICKS_TOKEN`,
and `DATABRICKS_HTTP_PATH` so the app can read from `dev.safety_stock_gold`.